Tujuan: 
1. Memindahkan berkas partisi DDoS yang sudah ada ke folder terisolasi:
   dataset/processed/unseen_attack_test_parts/DDoS/
2. Mengekstrak 4 kategori serangan tambahan dari NF-UQ-NIDS-v2.csv secara streaming:
   - DoS
   - Reconnaissance
   - Brute Force
   - Bot
   dan menyimpannya ke folder masing-masing secara terpisah.
Catatan Metodologi:
- Data Benign Train dan Benign Test TIDAK diproses ulang.
- Mempertahankan konsistensi 39 fitur numerik murni.

In [2]:
import json
import os
import shutil
from pathlib import Path
from typing import Dict, List
import numpy as np
import pandas as pd

# ==============================================================================
# 1. KONFIGURASI DIREKTORI & TARGET SERANGAN
# ==============================================================================
BASE_DIR = Path.cwd().resolve()
RAW_CSV = Path(r"../dataset/raw/NF-UQ-NIDS-v2.csv")
OUTPUT_DIR = Path(r"../dataset/processed")
UNSEEN_ATTACK_DIR = OUTPUT_DIR / "unseen_attack_test_parts"

META_PATH = OUTPUT_DIR / "partition_metadata.json"
LABEL_COLUMN = "Label"
ATTACK_COLUMN = "Attack"
CHUNKSIZE = 50_000
LIMIT_CHUNKS = None  # Pasang None untuk memproses seluruh data, atau misal 100 untuk uji coba

# Pemetaan 5 Kategori Serangan Target untuk Unseen Zero-Day Evaluation
TARGET_ATTACKS: Dict[str, str] = {
    "ddos": "DDoS",
    "dos": "DoS",
    "reconnaissance": "Reconnaissance",
    "brute force": "Brute Force",
    "bot": "Bot"
}

# Verifikasi keberadaan metadata fitur Part 1 & 2
assert META_PATH.exists(), f"Metadata tidak ditemukan di {META_PATH}! Pastikan Part 1 & 2 telah selesai."
with open(META_PATH, "r", encoding="utf-8") as f:
    meta_info = json.load(f)
FEATURE_COLUMNS = meta_info["feature_columns"]

print("=" * 80)
print("SISTEM PARTISI MULTI-ATTACK PER FOLDER (NF-UQ-NIDS-v2)")
print("=" * 80)
print("Direktori Berkas Mentah :", RAW_CSV.resolve())
print("Direktori Target Output :", UNSEEN_ATTACK_DIR.resolve())
print(f"Jumlah Fitur Masukan    : {len(FEATURE_COLUMNS)} fitur numerik")
print("Target 5 Serangan       :", list(TARGET_ATTACKS.values()))

SISTEM PARTISI MULTI-ATTACK PER FOLDER (NF-UQ-NIDS-v2)
Direktori Berkas Mentah : I:\skripsi\NIDS\dataset\raw\NF-UQ-NIDS-v2.csv
Direktori Target Output : I:\skripsi\NIDS\dataset\processed\unseen_attack_test_parts
Jumlah Fitur Masukan    : 39 fitur numerik
Target 5 Serangan       : ['DDoS', 'DoS', 'Reconnaissance', 'Brute Force', 'Bot']


In [3]:
# ==============================================================================
# 2. LANGKAH 1: MIGRASI PARTISI DDOS LAMA KE SUBFOLDER 'DDoS/'
# ==============================================================================
def migrate_existing_ddos_parts():
    """
    Memindahkan berkas Parquet DDoS yang berada langsung di root folder
    unseen_attack_test_parts ke dalam subdirektori unseen_attack_test_parts/DDoS/
    """
    ddos_dir = UNSEEN_ATTACK_DIR / "DDoS"
    ddos_dir.mkdir(parents=True, exist_ok=True)

    existing_files = [f for f in UNSEEN_ATTACK_DIR.glob("*.parquet") if f.is_file()]
    
    if len(existing_files) > 0:
        print(f"\n[MIGRASI] Ditemukan {len(existing_files)} berkas Parquet DDoS lama.")
        print(f"Memindahkan berkas ke folder terisolasi: {ddos_dir} ...")
        moved_count = 0
        for f in existing_files:
            target_file = ddos_dir / f.name
            shutil.move(str(f), str(target_file))
            moved_count += 1
        print(f"[SUKSES] {moved_count} berkas DDoS berhasil dipindahkan ke {ddos_dir.name}/.")
    else:
        current_ddos_in_sub = list(ddos_dir.glob("*.parquet"))
        print(f"\n[INFO] Folder {ddos_dir.name}/ telah siap dengan {len(current_ddos_in_sub)} berkas Parquet.")

migrate_existing_ddos_parts()



[MIGRASI] Ditemukan 1520 berkas Parquet DDoS lama.
Memindahkan berkas ke folder terisolasi: ..\dataset\processed\unseen_attack_test_parts\DDoS ...
[SUKSES] 1520 berkas DDoS berhasil dipindahkan ke DDoS/.


In [4]:

# ==============================================================================
# 3. FUNGSI UTILITAS PEMBERSIHAN DATA & SERIALISASI PARQUET
# ==============================================================================
def normalize_label_series(series: pd.Series, benign_val: int = 0) -> np.ndarray:
    """Mengonversi label menjadi format biner standar: 0 (Normal), 1 (Anomali)."""
    if pd.api.types.is_numeric_dtype(series):
        return (series != benign_val).astype("int8")
    s = series.astype(str).str.strip().str.lower()
    benign_set = {"benign", "normal", "0", "false", "no"}
    is_benign = s.isin(benign_set) | s.str.contains("benign", regex=False) | s.str.contains("normal", regex=False)
    return np.where(is_benign, 0, 1).astype("int8")

def clean_chunk_features(chunk: pd.DataFrame, features: List[str]) -> pd.DataFrame:
    """Membersihkan fitur numerik: konversi float32, eliminasi NaN dan tak hingga (inf)."""
    X = chunk[features].apply(lambda c: pd.to_numeric(c, errors="coerce"))
    X.replace([np.inf, -np.inf], np.nan, inplace=True)
    valid_idx = X.dropna().index
    if len(valid_idx) == 0:
        return None
    cleaned = chunk.loc[valid_idx].copy()
    cleaned[features] = X.loc[valid_idx].astype("float32")
    return cleaned

def save_attack_part(df: pd.DataFrame, attack_canonical_name: str, chunk_idx: int) -> int:
    """Menyimpan potongan DataFrame serangan ke berkas Parquet dalam subfolder kategori."""
    if df is None or df.empty:
        return 0
    target_folder = UNSEEN_ATTACK_DIR / attack_canonical_name
    target_folder.mkdir(parents=True, exist_ok=True)
    file_path = target_folder / f"part_{chunk_idx:06d}.parquet"
    df.to_parquet(file_path, index=False, compression="snappy")
    return len(df)

In [5]:

# ==============================================================================
# 4. LANGKAH 2: EKSTRAKSI STREAMING 4 SERANGAN BARU (DoS, Recon, Brute Force, Bot)
# ==============================================================================
# Saring 4 kategori serangan baru (DDoS dilewati karena sudah lengkap di folder DDoS/)
NEW_ATTACK_KEYS = {"dos", "reconnaissance", "brute force", "bot"}

print("\n" + "=" * 80)
print("MEMULAI EKSTRAKSI STREAMING UNTUK SERANGAN BARU DARI CSV MENTAH")
print("Target Ekstraksi Baru: DoS, Reconnaissance, Brute Force, Bot")
print("=" * 80)

load_columns = FEATURE_COLUMNS + [LABEL_COLUMN, ATTACK_COLUMN]
counters = {
    "rows_scanned": 0,
    "DoS": 0,
    "Reconnaissance": 0,
    "Brute Force": 0,
    "Bot": 0,
    "other_ignored": 0
}

reader = pd.read_csv(
    RAW_CSV,
    usecols=load_columns,
    chunksize=CHUNKSIZE,
    sep=",",
    low_memory=False,
)

for chunk_idx, chunk in enumerate(reader):
    if LIMIT_CHUNKS is not None and chunk_idx >= LIMIT_CHUNKS:
        print(f"\n[INFO] Batas LIMIT_CHUNKS ({LIMIT_CHUNKS}) tercapai. Menghentikan proses.")
        break

    counters["rows_scanned"] += len(chunk)

    # Saring hanya baris berlabel serangan (Label == 1)
    chunk = chunk.dropna(subset=[LABEL_COLUMN])
    if chunk.empty:
        continue

    labels = normalize_label_series(chunk[LABEL_COLUMN])
    attack_mask = (labels == 1)
    if not attack_mask.any():
        continue

    attack_chunk = chunk[attack_mask].copy()

    # Normalisasi string nama serangan
    attack_chunk[ATTACK_COLUMN] = attack_chunk[ATTACK_COLUMN].fillna("Unknown").astype(str).str.strip()
    attack_chunk["attack_lower"] = attack_chunk[ATTACK_COLUMN].str.lower()

    # Filter hanya 4 jenis serangan yang ditargetkan
    is_target_attack = attack_chunk["attack_lower"].isin(NEW_ATTACK_KEYS)
    target_data = attack_chunk[is_target_attack]
    counters["other_ignored"] += len(attack_chunk) - len(target_data)

    if target_data.empty:
        continue

    # Bersihkan fitur numerik
    cleaned_target = clean_chunk_features(target_data, FEATURE_COLUMNS)
    if cleaned_target is None or cleaned_target.empty:
        continue

    # Simpan per subfolder kategori serangan masing-masing
    for atk_lower, group in cleaned_target.groupby("attack_lower"):
        canonical_name = TARGET_ATTACKS[atk_lower]
        saved_rows = save_attack_part(
            df=group[FEATURE_COLUMNS + [LABEL_COLUMN, ATTACK_COLUMN]],
            attack_canonical_name=canonical_name,
            chunk_idx=chunk_idx
        )
        counters[canonical_name] += saved_rows

    if chunk_idx % 40 == 0:
        print(
            f"Chunk {chunk_idx:4d} | Terpindai: {counters['rows_scanned']:,} | "
            f"DoS: {counters['DoS']:,} | Recon: {counters['Reconnaissance']:,} | "
            f"BruteForce: {counters['Brute Force']:,} | Bot: {counters['Bot']:,}"
        )

print("\n" + "=" * 80)
print("PROSES EKSTRAKSI MULTI-ATTACK SELESAI!")
print("=" * 80)
print("Ringkasan Aliran Serangan Baru:")
print(f"- DoS            : {counters['DoS']:,} aliran")
print(f"- Reconnaissance : {counters['Reconnaissance']:,} aliran")
print(f"- Brute Force    : {counters['Brute Force']:,} aliran")
print(f"- Bot            : {counters['Bot']:,} aliran")



MEMULAI EKSTRAKSI STREAMING UNTUK SERANGAN BARU DARI CSV MENTAH
Target Ekstraksi Baru: DoS, Reconnaissance, Brute Force, Bot
Chunk    0 | Terpindai: 50,000 | DoS: 11,766 | Recon: 1,728 | BruteForce: 98 | Bot: 111
Chunk   40 | Terpindai: 2,050,000 | DoS: 482,704 | Recon: 70,551 | BruteForce: 3,437 | Bot: 3,905
Chunk   80 | Terpindai: 4,050,000 | DoS: 953,447 | Recon: 140,022 | BruteForce: 6,653 | Bot: 7,705
Chunk  120 | Terpindai: 6,050,000 | DoS: 1,424,399 | Recon: 209,133 | BruteForce: 9,923 | Bot: 11,542
Chunk  160 | Terpindai: 8,050,000 | DoS: 1,894,130 | Recon: 278,275 | BruteForce: 13,307 | Bot: 15,276
Chunk  200 | Terpindai: 10,050,000 | DoS: 2,363,733 | Recon: 347,893 | BruteForce: 16,569 | Bot: 19,011
Chunk  240 | Terpindai: 12,050,000 | DoS: 2,834,536 | Recon: 417,177 | BruteForce: 19,767 | Bot: 22,847
Chunk  280 | Terpindai: 14,050,000 | DoS: 3,304,819 | Recon: 486,755 | BruteForce: 22,998 | Bot: 26,665
Chunk  320 | Terpindai: 16,050,000 | DoS: 3,775,590 | Recon: 555,946 | B

In [6]:
# ==============================================================================
# 5. REKAPITULASI STRUKTUR FOLDER & METADATA
# ==============================================================================
summary_info = {}
print("\nStruktur Direktori 'dataset/processed/unseen_attack_test_parts/':")
for attack_name in sorted(list(set(TARGET_ATTACKS.values()))):
    subfolder = UNSEEN_ATTACK_DIR / attack_name


Struktur Direktori 'dataset/processed/unseen_attack_test_parts/':
